# Customer Churn Prediction Using Machine Learning
This notebook follows the full data mining pipeline required for the final project.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 3. Data Understanding

In [ ]:
print(df.shape)
print(df.info())
print(df['Churn'].value_counts())

## 4. Data Cleaning and Preprocessing

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
y = df['Churn'].map({'No':0, 'Yes':1})
X = df.drop(columns=['Churn'])
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))]), cat_cols)
])

## 5. Sampling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

## 6. Decision Tree and SVM Models

In [ ]:
dt_model = Pipeline([('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'))])
svm_model = Pipeline([('preprocessor', preprocessor), ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))])
dt_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)

## 7. Evaluation

In [ ]:
def evaluate(name, model):
    pred = model.predict(X_test)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred)
    }
results = pd.DataFrame([evaluate('Decision Tree', dt_model), evaluate('SVM', svm_model)])
results

## 8. K-Means Clustering

In [ ]:
X_processed = preprocessor.fit_transform(X_train)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_processed)
print('Clusters:', np.unique(clusters, return_counts=True))